In [8]:
import numpy as np, pandas as pd
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.model_selection import train_test_split

paired = pd.read_csv("/content/standup_fusion_probs.csv")
train_df = paired[paired.split=="train"].reset_index(drop=True)
test_df = paired[paired.split=="test"].reset_index(drop=True)
print(len(train_df), len(test_df))

test_patients = test_df["patient_id"].unique()
val_pid, final_pid = train_test_split(test_patients, test_size=0.5, random_state=42)

weight_val_df = test_df[test_df["patient_id"].isin(val_pid)].reset_index(drop=True)
final_test_df = test_df[test_df["patient_id"].isin(final_pid)].reset_index(drop=True)
print(len(weight_val_df), len(final_test_df))

331 84
41 43


In [9]:
def fuse_probs(df, w):
    fused = {}
    for k in range(3):
        fused[k] = w*df[f"p_rgb_{k}"].values + (1-w)*df[f"p_thermal_{k}"].values
    return np.stack([fused[0], fused[1], fused[2]], axis=1)

def ordinal_predict(probs):
    return (probs > 0.5).sum(axis=1)

def score_weight(w, df, metric="qwk"):
    probs = fuse_probs(df, w)
    preds = ordinal_predict(probs)
    y = df["ordinal_label"].values
    if metric == "qwk":
        return cohen_kappa_score(y, preds, weights="quadratic")
    return accuracy_score(y, preds)

In [10]:
grid_vals = np.arange(0.0, 1.01, 0.05)
best_grid_w, best_grid_qwk = None, -1

for w in grid_vals:
    q = score_weight(w, weight_val_df, "qwk")
    if q > best_grid_qwk:
        best_grid_qwk, best_grid_w = q, w

print(f"Grid search best w_rgb={best_grid_w:.2f} (w_thermal={1-best_grid_w:.2f}) | Train QWK={best_grid_qwk:.4f}")

Grid search best w_rgb=0.60 (w_thermal=0.40) | Train QWK=0.7539


In [11]:
def sigmoid(z): return 1/(1+np.exp(-z))

def neg_qwk_from_z(z, df):
    w = sigmoid(z[0])
    return -score_weight(w, df, "qwk")

z0 = np.array([np.log(best_grid_w/(1-best_grid_w+1e-8)+1e-8)])
res = minimize(neg_qwk_from_z, z0, args=(weight_val_df,), method="Nelder-Mead",
                options={"xatol":1e-6, "fatol":1e-6, "maxiter":2000})

refined_w = sigmoid(res.x[0])
refined_qwk = score_weight(refined_w, weight_val_df, "qwk")
final_w = refined_w if refined_qwk > best_grid_qwk else best_grid_w
print(f"Refined w_rgb={refined_w:.4f} | Train QWK={refined_qwk:.4f}")
print(f"Final chosen w_rgb={final_w:.4f}")

Refined w_rgb=0.6000 | Train QWK=0.7539
Final chosen w_rgb=0.6000


In [12]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_weights, fold_qwks = [], []

for fold, (tr_idx, val_idx) in enumerate(sgkf.split(weight_val_df, weight_val_df["ordinal_label"], groups=weight_val_df["patient_id"]), 1):
    tr_fold, val_fold = weight_val_df.iloc[tr_idx], weight_val_df.iloc[val_idx]
    best_w, best_q = None, -1
    for w in grid_vals:
        q = score_weight(w, tr_fold, "qwk")
        if q > best_q:
            best_q, best_w = q, w
    val_qwk = score_weight(best_w, val_fold, "qwk")
    fold_weights.append(best_w); fold_qwks.append(val_qwk)
    print(f"Fold {fold}: w_rgb={best_w:.2f} | Val QWK={val_qwk:.4f}")

print(f"\nMean w_rgb={np.mean(fold_weights):.4f} +/- {np.std(fold_weights):.4f}")
print(f"Mean Val QWK={np.mean(fold_qwks):.4f} +/- {np.std(fold_qwks):.4f}")

Fold 1: w_rgb=0.60 | Val QWK=0.8756
Fold 2: w_rgb=0.55 | Val QWK=0.2326
Fold 3: w_rgb=0.60 | Val QWK=0.8372
Fold 4: w_rgb=0.85 | Val QWK=0.6429
Fold 5: w_rgb=0.60 | Val QWK=0.8182

Mean w_rgb=0.6400 +/- 0.1068
Mean Val QWK=0.6813 +/- 0.2382


In [14]:
final_probs = fuse_probs(final_test_df, final_w)
final_preds = ordinal_predict(final_probs)

results = []
for name, col in [("RGB only","p_rgb"), ("Thermal only","p_thermal")]:
    preds = ordinal_predict(np.stack([final_test_df[f"{col}_{k}"] for k in range(3)], axis=1))
    results.append({"name": name, "accuracy": accuracy_score(final_test_df["ordinal_label"], preds),
                     "qwk": cohen_kappa_score(final_test_df["ordinal_label"], preds, weights="quadratic")})
results.append({"name": f"Fused (w_rgb={final_w:.3f})",
                 "accuracy": accuracy_score(final_test_df["ordinal_label"], final_preds),
                 "qwk": cohen_kappa_score(final_test_df["ordinal_label"], final_preds, weights="quadratic")})

results_df = pd.DataFrame(results)
results_df.to_csv("/content/standup_final_fusion_results.csv", index=False)
print(results_df)

                  name  accuracy       qwk
0             RGB only  0.441860  0.484054
1         Thermal only  0.372093  0.328902
2  Fused (w_rgb=0.600)  0.488372  0.522099
